<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="40%"></a>
</p>

<span style="float:right">
<strong>Credits:</strong>
Ce tutoriel est une adaptation du <a href="https://wiki.ros.org/ROS/Tutorials/">tutoriel officiel de ROS</a>, initialement publié sous la licence <a href="http://creativecommons.org/licenses/by/3.0/">Creative Commons Attribution 3.0</a>.<span>

# ROS Topics

Ce tutoriel présente les ROS topic ainsi que l'utilisation des outils en ligne de commande [rostopic](http://wiki.ros.org/rostopic) et [rqt_plot](http://wiki.ros.org/rqt_plot).

## Suivre le tutoriel

Comme pour le notebook précédent, vous aurez besoin d'un environnement graphique connecté à votre robot.
Pour le lancer, vous pouvez exécuter la commande suivante dans le terminal **de votre ordinateur portable** (et pas dans le terminal de ce `VSCode` !) :

    dts code vnc -R [!ROBOT_NAME] 

Cela lancera un conteneur avec ROS installé. Un lien apparaîtra dans le terminal et, en cliquant dessus, vous accéderez à un environnement VNC Desktop.


Sur le Desktop, vous trouverez plusieurs icônes ; pour ouvrir un terminal, vous pouvez utiliser `LXTerminal`.

![desktop_icons](../assets/desktop_icons.png)



### turtlesim

Pour ce tutoriel, nous utiliserons également turtlesim. S'il est toujours en cours d'exécution depuis le précédent notebook, vous pouvez ignorer cette étape ; sinon, veuillez exécuter la commande suivante :


    rosrun turtlesim turtlesim_node


### téléopération du clavier de la tortue

Nous aurons également besoin de quelque chose pour contrôler la tortue. Veuillez exécuter la commande suivante **dans un nouveau terminal** (sur le VNC Desktop) :

```
rosrun turtlesim turtle_teleop_key
```

  ```
  Reading from keyboard
  ---------------------------
  Use arrow keys to move the turtle.
  ```
    

Vous pouvez maintenant utiliser les touches fléchées du clavier pour déplacer la tortue. Si vous ne parvenez pas à la déplacer, sélectionnez la fenêtre du terminal `turtle_teleop_key` pour vous assurer que les touches que vous appuyez sont bien enregistrées. 

![turtle_key](../assets/turtle_key.png)

Maintenant que vous pouvez déplacer votre tortue, voyons ce qui se passe en coulisses.


## ROS Topics

Les nodes `turtlesim_node` et `turtle_teleop_key` communiquent entre eux via un  ROS **topic**. `turtle_teleop_key` **publishes** les touches pressées sur un sujet, tandis que turtlesim **subscribes** à ce même sujet pour recevoir ces informations. Vous pouvez utiliser [rqt_graph](http://wiki.ros.org/rqt_graph), qui affiche les nœuds et les sujets actuellement en cours d'exécution.

```
rqt_graph
```

### `rostopic`

L'outil `rostopic` permet d'obtenir des informations sur les ROS **topics**.

Vous pouvez utiliser l'option d'aide pour obtenir la liste des sous-commandes disponibles pour `rostopic`.

```
rostopic -h
```

-   ```
    rostopic bw     display bandwidth used by topic
    rostopic echo   print messages to screen
    rostopic hz     display publishing rate of topic    
    rostopic list   print information about active topics
    rostopic pub    publish data to topic
    rostopic type   print topic type
    ```
    

Utilisons certaines de ces sous-commandes relatives aux sujets pour examiner turtlesim.

### `rostopic echo`

La commande `rostopic echo` affiche les données publiées sur un sujet.

Usage:

```
rostopic echo [topic]
```

Examinons les données de vitesse de commande publiées par le node `turtle_teleop_key`.

Ces données sont publiées sur le topic `turtle1/cmd_vel`. **Dans un nouveau terminal**, exécutez la commande suivante :

```
rostopic echo /turtle1/cmd_vel
```


Vous ne verrez probablement rien se passer car aucune donnée n'est publiée sur ce sujet. Faisons en sorte que `turtle_teleop_key` publish des données lorsque vous appuyez sur les touches fléchées. *N'oubliez pas que si la tortue ne bouge pas, vous devez sélectionner à nouveau la fenêtre du terminal `turtle_teleop_key`.*

Vous devriez maintenant voir ceci lorsque vous appuyez sur la flèche vers le haut :

```
linear: 
  x: 2.0
  y: 0.0
  z: 0.0
angular: 
  x: 0.0
  y: 0.0
  z: 0.0
---
linear: 
  x: 2.0
  y: 0.0
  z: 0.0
angular: 
  x: 0.0
  y: 0.0
  z: 0.0
---
```

### `rostopic list`

La commande `rostopic list` affiche la liste de tous les topics auxquels on est actuellement subscribed et de ceux qui sont published.

Dans un **nouveau terminal**, exécutez:

  rostopic list -h

  ```
  Usage: rostopic list [/topic]
  
  Options:
    -h, --help            show this help message and exit
    -b BAGFILE, --bag=BAGFILE
                          list topics in .bag file
    -v, --verbose         list full details about each topic
    -p                    list only publishers
    -s                    list only subscribers
  ```
    

Pour la commande rostopic list, utilisez l'option **verbose** :

  rostopic list -v

Ceci affiche une liste détaillée des topics auxquels il est possible de publish et de subscribe, ainsi que leur type. Pour filtrer et n'afficher que les sujets liés à notre exemple turtlesim, procédez comme suit :

```
rostopic list -v | grep turtle

```


  ```
  * /turtle1/pose [turtlesim/Pose] 1 publisher
 * /turtle1/color_sensor [turtlesim/Color] 1 publisher
 * /turtle1/cmd_vel [geometry_msgs/Twist] 1 publisher
 * /turtle1/cmd_vel [geometry_msgs/Twist] 1 subscriber

  ```
    
## ROS Messages

La communication sur les différents topics s'effectue par l'envoi de **messages** ROS entre les nodes. Pour qu'un publisher (turtle_teleop_key) et un subsscriber (turtlesim_node) puissent communiquer, ils doivent envoyer et recevoir le même **type** de message. Cela signifie que le **type** d'un topic est défini par le **type** de message qui y est published. Le **type** du message envoyé sur un topic peut être déterminé à l'aide de la commande `rostopic type`.

### `rostopic type`

La commande `rostopic type` renvoie le type de message de n'importe quel topic qui est published.

Usage:

```
rostopic type [topic]
```

-   Try:
    
    ```
    rostopic type /turtle1/cmd_vel
    ```
    
    -   You should get:
        
        ```
        geometry_msgs/Twist
        ```
        
    
    We can look at the details of the message using rosmsg:
    
    ```
    rosmsg show geometry_msgs/Twist
    ```
    
    -   ```
        geometry_msgs/Vector3 linear
          float64 x
          float64 y
          float64 z
        geometry_msgs/Vector3 angular
          float64 x
          float64 y
          float64 z
        ```
      

Maintenant que nous savons quel type de message turtlesim attend, nous pouvons publish des commandes à notre tortue.

## `rostopic` continué

Maintenant que nous avons découvert les **messages** de ROS, utilisons la commande `rostopic` avec ces messages.

### `rostopic pub`

La commande `rostopic pub` publishes des messages contenant des données sur un topic actuellement annoncé.

Usage:

```
rostopic pub [topic] [msg_type] [args]
```

```
rostopic pub -1 /turtle1/cmd_vel geometry_msgs/Twist -- '[2.0, 0.0, 0.0]' '[0.0, 0.0, 1.8]'
```

La commande précédente enverra un seul message à turtlesim lui indiquant de se déplacer avec une vitesse linéaire de 2,0 et une vitesse angulaire de 1,8.

![turtle(rostopicpub).png](../assets/turtle(rostopicpub).png)
    

Il s'agit d'un exemple assez complexe, alors examinons chaque argument en détail.

-   Cette commande permettra de publier des messages sur un topic donné :
    
    ```
    rostopic pub
    ```
    
-   Cette option (`-1`) permet à rostopic de publish un seul message, puis de se terminer :
    
    ```
     -1 
    ```
    
-   Voici le nom du topic sur lequel publish :
    
    ```
    /turtle1/cmd_vel
    ```
    
-   Il s'agit du type de message à utiliser lors de la publish sur le topic:
    
    ```
    geometry_msgs/Twist
    ```
    
-   Cette option (`--`) indique à l'analyseur d'options qu'aucun des arguments suivants n'est une option. Ceci est nécessaire dans les cas où vos arguments commencent par un tiret (-), comme pour les nombres négatifs.
    
    ```
    --
    ```
    
-  Comme mentionné précédemment, un message `geometry_msgs/Twist` contient deux vecteurs de trois éléments à virgule flottante chacun : linéaire et angulaire. Dans ce cas, `[2.0, 0.0, 0.0]` correspond à la valeur linéaire avec x=2.0, y=0.0 et z=0.0, et `[0.0, 0.0, 1.8]` à la valeur angulaire avec x=0.0, y=0.0 et z=1.8. Ces arguments sont en réalité au format YAML, décrit plus en détail dans la [documentation de la ligne de commande YAML](http://wiki.ros.org/ROS/YAMLCommandLine).    

    ```
    '[2.0, 0.0, 0.0]' '[0.0, 0.0, 1.8]' 
    ```
    

Vous avez peut-être remarqué que la tortue a cessé de bouger ; c'est parce qu'elle a besoin d'un flux constant de commandes à une fréquence de 1 Hz pour continuer à se déplacer. Nous pouvons publish un flux constant de commandes à l'aide de la commande `rostopic pub -r` :

-   ```
    rostopic pub /turtle1/cmd_vel geometry_msgs/Twist -r 1 -- '[2.0, 0.0, 0.0]' '[0.0, 0.0, -1.8]'
    ```


Cela publish les commandes de vitesse à une fréquence de 1 Hz sur le sujet de la vitesse.

![turtle(rostopicpub)2.png](../assets/turtle(rostopicpub)2.png)
    


Comme vous pouvez le voir, la tortue se déplace en cercle continu. Dans un **nouveau terminal**, nous pouvons utiliser la commande `rostopic echo` pour visualiser les données publiées par notre simulateur de tortue (turtlesim) :

```
rostopic echo /turtle1/pose
```

### `rostopic hz`

`rostopic hz` indique la fréquence à laquelle les données sont published.

Usage:

```
rostopic hz [topic]
```

Voyons à quelle vitesse le nœud turtlesim_node publish les données sur /turtle1/pose :

```
rostopic hz /turtle1/pose
```

You will see:

-   ```
    subscribed to [/turtle1/pose]
    average rate: 59.354
            min: 0.005s max: 0.027s std dev: 0.00284s window: 58
    average rate: 59.459
            min: 0.005s max: 0.027s std dev: 0.00271s window: 118
    average rate: 59.539
            min: 0.004s max: 0.030s std dev: 0.00339s window: 177
    average rate: 59.492
            min: 0.004s max: 0.030s std dev: 0.00380s window: 237
    average rate: 59.463
            min: 0.004s max: 0.030s std dev: 0.00380s window: 290
    ```
    

Nous pouvons maintenant constater que turtlesim publish des données concernant notre tortue à une fréquence de 60 Hz. Nous pouvons également utiliser la commande `rostopic type` conjointement avec `rosmsg show` pour obtenir des informations détaillées sur un topic donné :

-   ```
    rostopic type /turtle1/cmd_vel | rosmsg show
    ```

Maintenant que nous avons examiné les topics à l'aide de `rostopic`, utilisons un autre outil pour visualiser les données publiées par notre simulateur turtlesim :

## `rqt_plot`

`rqt_plot` affiche un graphique temporel défilant des données pubished sur les topics. Nous allons utiliser `rqt_plot` pour afficher les données published sur le topic `/turtle1/pose`. Commencez par lancer `rqt_plot` en tapant :

```
rosrun rqt_plot rqt_plot
```

Dans la nouvelle fenêtre qui s'affiche, une zone de texte en haut à gauche vous permet d'ajouter des topics au graphique. En tapant `/turtle1/pose/x`, le bouton « plus », auparavant désactivé, s'active. Cliquez dessus et répétez la même procédure avec le topic `/turtle1/pose/y`. Vous verrez alors la position x-y de la tortue tracée sur le graphique.


![rqt_plot.png](../assets/images/rqt_plot.png)

Appuyer sur le bouton moins affiche un menu qui permet de masquer le topic spécifié dans le graphique. Masquer les deux topics que vous venez d'ajouter et ajouter `/turtle1/pose/theta` donnera le graphique présenté dans la figure suivante.

![rqt_plot2.png](../assets/images/rqt_plot2.png)

C'est tout pour cette section. Utilisez `Ctrl-C` pour fermer les terminaux `rostopic`, mais laissez turtlesim fonctionner.

Nous allons examiner comment définir nos propres messages ROS dans [le prochain notebook](../notebooks/06_ros_messages.ipynb).